# Gold Layer

Prepare data for analysis and machine learning algorithms by aggregating and feature engineering.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

def read_silver():
    df = spark.read.format("delta").load("/Volumes/workspace/default/silver/energy_clean")
    return df

def build_hourly_features(df):
    # 5 min intervals to hourly buckets
    df = df.withColumn("hour_ts", F.date_trunc("hour", "settlement_ts"))

    # sum demand and price by hours
    df = df.groupBy("hour_ts", "region").agg(
        F.mean("total_demand_mw").alias("avg_demand_mw"),
        F.max("total_demand_mw").alias("max_demand_mw"),
        F.mean("price_rrp").alias("avg_price_rrp")
    )

    # time features for ml
    df = df.withColumn("date", F.to_date("hour_ts"))
    df = df.withColumn("hour", F.hour("hour_ts").cast(IntegerType()))
    df = df.withColumn("day_of_week", F.dayofweek("hour_ts").cast(IntegerType()))  # 1 = sun, 7 = sat
    df = df.withColumn("month", F.month("hour_ts").cast(IntegerType()))
    df = df.withColumn("quarter", F.quarter("hour_ts").cast(IntegerType()))
    df = df.withColumn("year", F.year("hour_ts").cast(IntegerType()))
    df = df.withColumn("is_weekend", (F.col("day_of_week").isin([1, 7])).cast(IntegerType()))

    # derive season from month for southern hemisphere
    df = df.withColumn("season", F.when(F.col("month").isin([12, 1, 2]), "summer")
                                  .when(F.col("month").isin([3, 4, 5]), "autumn")
                                  .when(F.col("month").isin([6, 7, 8]), "winter")
                                  .otherwise("spring"))

    df = df.orderBy("hour_ts")
    return df

def write_gold_features(df):
    (df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save("/Volumes/workspace/default/gold/demand_hourly"))
    print("[GOLD] hourly features written")

def write_gold_forecast_placeholder(df):
    # create empty forecast table
    forecast_df = (df
        .select("hour_ts", "region", F.col("avg_demand_mw").alias("actual_demand_mw"))
        .limit(0)
        .withColumn("predicted_demand_mw", F.lit(None).cast("double"))
        .withColumn("model_version", F.lit("v0"))
        .withColumn("created_at", F.current_timestamp())
    )

    (forecast_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save("/Volumes/workspace/default/gold/demand_forecast"))
    print("[GOLD] forecast placeholder written")

def main(df = build_hourly_features(read_silver())):
    write_gold_features(df)
    write_gold_forecast_placeholder(df)

## Aggregation

Run the preview block before `main` to execute the final layer.

### Running

Create the gold layer dataframe and check the schema before writing it to volumes.

In [0]:
df_silver = read_silver()
df_gold = build_hourly_features(df_silver)
df_gold.printSchema()
df_gold.show(5)

In [0]:
main(df_gold)